<a href="https://colab.research.google.com/github/AbhiSethiya/GoogleCamp/blob/main/C7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os


In [ ]:
import kagglehub

# Download latest version and define KAGGLE_DATASET_PATH
KAGGLE_DATASET_PATH = kagglehub.dataset_download("samuelayman/bird-dataset")
print("Path to dataset files:", KAGGLE_DATASET_PATH)

# Step 1: Define constants
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50

train_dir = os.path.join(KAGGLE_DATASET_PATH, 'train')
val_dir = os.path.join(KAGGLE_DATASET_PATH, 'validation')

Using Colab cache for faster access to the 'bird-dataset' dataset.
Path to dataset files: /kaggle/input/bird-dataset


In [ ]:
import kagglehub
import os

# Download latest version
KAGGLE_DATASET_PATH = kagglehub.dataset_download("samuelayman/bird-dataset")

print("Path to dataset files:", KAGGLE_DATASET_PATH)

Using Colab cache for faster access to the 'bird-dataset' dataset.
Path to dataset files: /kaggle/input/bird-dataset


In [ ]:
print(f"Contents of {train_dir}:")
for root, dirs, files in os.walk(train_dir):
    level = root.replace(train_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')


Contents of /kaggle/input/bird-dataset/train:


In [ ]:
# Step 2: Data generators with augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
val_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Step 3: Load images
# The actual dataset directories will now be used after correcting train_dir and val_dir.
# Dummy directory creation is no longer needed.

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print(f"Found {train_generator.num_images} images belonging to {train_generator.num_classes} classes in training set.")
print(f"Found {val_generator.num_images} images belonging to {val_generator.num_classes} classes in validation set.")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/bird-dataset/train'

In [ ]:
# Step 4: Load ResNet50 base (no top classifier)
base_model = ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base model


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Step 4 & 5: Initialize ResNet50 and add custom classification layers
base_model = ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base model

num_classes = train_generator.num_classes

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)
print("Model successfully initialized with base_model and custom top layers.")

In [ ]:
# Step 6: Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
# Step 7: Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)


In [ ]:
from PIL import Image
import numpy as np
import os

def create_dummy_image(filepath, size=(IMG_SIZE, IMG_SIZE)):
    img_array = np.zeros((size[0], size[1], 3), dtype=np.uint8)
    center_x, center_y = size[0] // 2, size[1] // 2
    square_size = min(size) // 4
    img_array[center_y - square_size:center_y + square_size,
              center_x - square_size:center_x + square_size] = 255
    img = Image.fromarray(img_array)
    img.save(filepath)

# We will use a local directory for dummy data to avoid read-only errors on Kaggle paths
local_train_dir = 'bird_dataset_local/train'
local_val_dir = 'bird_dataset_local/validation'

num_dummy_images_per_class = 5
classes = ['sparrow', 'eagle']

for base_path in [local_train_dir, local_val_dir]:
    for class_name in classes:
        class_dir = os.path.join(base_path, class_name)
        os.makedirs(class_dir, exist_ok=True)
        for i in range(num_dummy_images_per_class):
            image_filepath = os.path.join(class_dir, f'{class_name}_{i+1}.png')
            create_dummy_image(image_filepath)

# Update the directory variables for the generators
train_dir = local_train_dir
val_dir = local_val_dir

print(f"Created dummy images in {train_dir} and {val_dir}")

In [ ]:
# Re-run the flow_from_directory calls using the verified local paths
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:
# Step 8: Evaluate
loss, acc = model.evaluate(val_generator)
print(f"Validation Accuracy: {acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# Get one batch of images and labels from the training generator
images, labels = next(train_generator)

# Map one-hot encoded labels back to class names
class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(10, 10))
for i in range(min(9, images.shape[0])):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    # Get the class index from the one-hot encoded label
    class_idx = np.argmax(labels[i])
    plt.title(class_names[class_idx])
    plt.axis('off')
plt.suptitle("Sample Dummy Images from Training Generator", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os


In [ ]:
# 1. Define constants (Unified to match the ResNet50 requirements)
IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
# Removed redundant definition to maintain consistency with ResNet50 requirements (IMG_SIZE=224)
print(f"Using unified constants: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}")

In [ ]:
# Use the local directory where dummy images were generated
train_dir = "bird_dataset_local/train"
val_dir = "bird_dataset_local/validation"
print(f"Data directories set to: {train_dir} and {val_dir}")

In [ ]:
# 3. Define ImageDataGenerator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,               # Normalize pixel values
    rotation_range=40,            # Random rotation
    width_shift_range=0.2,        # Horizontal shift
    height_shift_range=0.2,       # Vertical shift
    shear_range=0.2,              # Shearing transformation
    zoom_range=0.2,               # Random zoom
    horizontal_flip=True,         # Flip horizontally
    fill_mode='nearest'           # Strategy for filling pixels
)


In [ ]:
# 4. Create the data generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


In [ ]:
# 5. Visualize some augmented images
images, labels = next(train_generator)  # Get one batch
class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(images[i])
    label_index = np.argmax(labels[i])
    plt.title(f"Class: {class_names[label_index]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------------------------
# 1. GENERATE DUMMY DATA (Object Detection Synthetic Task)
# Note: We keep IMG_SIZE=224 for consistency across the notebook
# ---------------------------------------------------------
NUM_IMAGES = 1000
print(f"Using IMG_SIZE: {IMG_SIZE} for synthetic data generation.")

In [ ]:
def create_synthetic_data(num_samples):
    X = np.zeros((num_samples, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    y = np.zeros((num_samples, 4), dtype=np.float32) # [x_min, y_min, width, height]

    for i in range(num_samples):
        # Random size and position for the "logo"
        w = np.random.randint(10, 40)
        h = np.random.randint(10, 40)
        x_min = np.random.randint(0, IMG_SIZE - w)
        y_min = np.random.randint(0, IMG_SIZE - h)

        # Draw the white square
        X[i, y_min:y_min+h, x_min:x_min+w, :] = 1.0

        # Normalize coordinates between 0 and 1 for the neural network
        y[i] = [x_min/IMG_SIZE, y_min/IMG_SIZE, w/IMG_SIZE, h/IMG_SIZE]

    return X, y

In [ ]:
print("Generating synthetic classroom data...")
X_train, y_train = create_synthetic_data(NUM_IMAGES)
X_test, y_test = create_synthetic_data(10) # 10 images for testing
print(f"Generated {NUM_IMAGES} training images.")

In [ ]:
# ---------------------------------------------------------
# 2. BUILD THE CNN ARCHITECTURE
# ---------------------------------------------------------
model = models.Sequential([
    # Convolutional Feature Extractor
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten the features into a 1D array
    layers.Flatten(),

    # Fully Connected Dense Layers
    layers.Dense(128, activation='relu'),

    # OUTPUT LAYER: 4 Neurons for [x_min, y_min, width, height]
    # We use 'sigmoid' because our target coordinates are normalized between 0 and 1
    layers.Dense(4, activation='sigmoid')
])

In [ ]:
# We use Mean Squared Error (MSE) because we are predicting continuous numbers (coordinates), not classes.
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# ---------------------------------------------------------
# 3. TRAIN THE MODEL
# ---------------------------------------------------------
print("\nTraining the CNN...")
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# ---------------------------------------------------------
# 4. TEST AND VISUALIZE PREDICTIONS
# ---------------------------------------------------------
print("\nMaking predictions on test data...")
predictions = model.predict(X_test)

In [ ]:
# Plotting the first 3 test results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    img = X_test[i].copy()
    # Denormalize predicted coordinates
    pred_x = int(predictions[i][0] * IMG_SIZE)
    pred_y = int(predictions[i][1] * IMG_SIZE)
    pred_w = int(predictions[i][2] * IMG_SIZE)
    pred_h = int(predictions[i][3] * IMG_SIZE)

    # Draw RED bounding box for the CNN's Prediction
    cv2.rectangle(img, (pred_x, pred_y), (pred_x + pred_w, pred_y + pred_h), (1, 0, 0), 2)

    axes[i].imshow(img)
    axes[i].set_title(f"Predicted Box (Red)")
    axes[i].axis('off')

plt.show()